# UKPN Data Centre Load Profile → 8760h (1-hour time step)

Converts the open **UK Power Networks – Data Centre Demand Profiles** dataset into the load profile of **a single data centre, hourly, full 8760h/year**, ready to use as an input for energy-system optimization research (sizing, dispatch, PV/BHESS/hydrogen models...).

🔗 Original data source: https://ukpowernetworks.opendatasoft.com/explore/assets/ukpn-data-centre-demand-profiles/

## 1. Problem statement

Unlike other load types such as residential, household, or hospital loads, a **real**, hourly, full-year load profile of **an actual data centre** is almost impossible to find publicly, or it comes at a cost. Most operators do not publish real metered power data, and most of the "data centre electricity consumption" figures reported in industry publications are not time series — they are aggregated estimates. I ran into this problem while working on my research at https://github.com/baotramsuri/bhess_optimalsizing. Even widely used energy-system optimization software such as HOMER has no built-in data for the data centre load type.

The best open dataset I found with **real metered (smart meter) power data** is published by **UK Power Networks (UKPN)** at

> https://ukpowernetworks.opendatasoft.com/explore/assets/ukpn-data-centre-demand-profiles/

It contains half-hourly (30-minute) utilisation data for **~100 anonymised data centres** within UKPN's licence area (South East England), starting from 2023.

However, the raw file cannot be used directly for energy-system optimization research because:

- It bundles **~100 sites** together in one file (over 1 million rows) — too heavy to filter/browse in Excel.
- The time step is **30 minutes**, not the 1-hour step used by most optimization models (LP/MILP, 8760-hour typical year, etc.).
- The published value is a **normalised utilisation ratio** (apparent power ÷ anonymised maximum import capacity), not an absolute power value (kW) — because UKPN anonymises each site's actual contracted capacity.
- Some sites have **incomplete** data (missing timestamps, duplicates, abnormal values), and UKPN itself recommends against using years with incomplete data.

Because of this, I had to build an intermediate conversion step to get the data in the form I actually needed. This notebook converts the raw UKPN file into **the load profile of one site, one year, full 8760 hours, in kW/kWh**, so that anyone doing energy-system optimization research can use a realistic data-centre load right away, without having to redo this entire data-cleaning process themselves.

## 2. General logic

The workflow has two stages, matching the two code sections below:

### Stage A — Find a site with sufficient data and extract it

1. Read the full UKPN file in chunks (to avoid loading over 1 million rows into memory at once).
2. For every `(anonymised data centre name, year)` pair, count:
   - total number of rows,
   - number of **unique** timestamps (duplicates removed).
3. Compare the unique-timestamp count against the **expected** number of half-hourly points for that year (`365 × 48 = 17,520`, or `366 × 48` for a leap year).
4. Flag each `(site, year)` pair as **complete** or not.
5. Print the list of sites with complete data for the target year — only these should be used to build an 8760h profile.
6. Once a site is chosen, extract that site's data for that year into a compact, standalone CSV file.

### Stage B — Convert to an hourly load profile

User-chosen inputs (set at the top of the script):

| Input | Example | Meaning |
|---|---|---|
| `INPUT_FILE` | `Data_Centre_#13_2025.csv` | Output of Stage A, i.e. the single-site CSV |
| `YEAR` | `2025` | Year to process — **must be a year UKPN itself flags as sufficiently complete**; do not force an incomplete year |
| `P_REF_MW` | `10.0` | Reference/assumed site capacity (MW), since UKPN anonymises the real maximum import capacity. This is a **modelling assumption chosen by the user**, not a measured value. |
| `POWER_FACTOR` | `0.95` | Assumed power factor, used to convert apparent power (kVA) to active power (kW) |

Processing logic:

1. Load the single-site CSV, keeping only `utc_timestamp` and `hh_utilisation_ratio`.
2. Filter to the requested `YEAR`.
3. **Data quality check** (strict — no automatic fixing):
   - missing half-hourly timestamps,
   - duplicate timestamps,
   - missing / negative utilisation values,
   - intervals between points that are not exactly 30 minutes.
   - If **any** issue is found, the script stops, writes a quality report, and does **not** interpolate or guess missing values — the user must pick a different site/year or handle the gaps explicitly.
4. If the data passes the check, **resample 30 minutes → 1 hour** by averaging the two half-hourly points within each hour (mathematically correct for a ratio/utilisation variable, unlike summation, which is only correct for an energy variable).
5. **Convert to physical units:**
   - Apparent power: `S(t) [kVA] = utilisation(t) × P_REF_MW × 1000`
   - Active power: `P(t) [kW] = S(t) × power_factor`
   - Energy: `E(t) [kWh] = P(t) × 1 h` (since time step = 1 hour)
6. Final check: the output must have exactly **8,760 hourly points** (non-leap-year assumption).
7. Export the hourly profile with columns: `hour, hourly_utilisation_ratio, apparent_power_kVA, load_kW, load_kWh`.
8. Print summary statistics: min/max/mean load, total annual energy (kWh and MWh), and load factor (mean ÷ max) — useful sanity checks before feeding the profile into a simulation model.

### Design choices worth noting

- **No interpolation of missing raw data.** If the chosen site/year has gaps, the script deliberately stops instead of filling values in — this keeps the resulting profile trustworthy for research use. The fix is to choose a cleaner site/year, not to modify the code to skip the check.
- **`P_REF_MW` is a user assumption, not a measured value**, because UKPN anonymises the real contracted capacity. When reusing this notebook, treat the absolute kW/kWh values as *scaled from a reference capacity you chose*, while the part that is genuinely derived from real measured data is the **shape** of the load over time.
- Utilisation values in the raw data can exceed 1.0 (observed power exceeding the registered import capacity for short periods) — the notebook keeps these values as-is rather than clipping them, so users can decide how to handle this for their own study.

## 3. Step by step (code)

### Steps 1–3: Stage A — scan and extract a single site

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. File
# ============================================================

BASE_DIR = Path.cwd()
csv_file = BASE_DIR / "ukpn-data-centre-demand-profiles.csv"

# ============================================================
# 2. Read only the required columns
# ============================================================

usecols = [
    "anonymised_data_centre_name",
    "utc_timestamp"
]

records = []

for chunk in pd.read_csv(
    csv_file,
    usecols=usecols,
    parse_dates=["utc_timestamp"],
    chunksize=100_000
):
    chunk["year"] = chunk["utc_timestamp"].dt.year

    # Count ROWS
    row_count = (
        chunk
        .groupby(["anonymised_data_centre_name", "year"])
        .size()
        .reset_index(name="n_rows")
    )

    # Count UNIQUE timestamps within this chunk
    unique_count = (
        chunk
        .drop_duplicates(
            ["anonymised_data_centre_name", "utc_timestamp"]
        )
        .groupby(["anonymised_data_centre_name", "year"])
        .size()
        .reset_index(name="n_unique_timestamps")
    )

    x = row_count.merge(
        unique_count,
        on=["anonymised_data_centre_name", "year"],
        how="outer"
    )

    records.append(x)

# ============================================================
# 3. Combine the chunks
# ============================================================

result = pd.concat(records, ignore_index=True)

rows = (
    result
    .groupby(
        ["anonymised_data_centre_name", "year"],
        as_index=False
    )["n_rows"]
    .sum()
)

# NOTE:
# Unique timestamp counts cannot simply be summed across chunks
# because the same timestamp could appear in different chunks.
# A second pass is therefore needed for an exact count.
# ============================================================

unique_timestamps = {}

for chunk in pd.read_csv(
    csv_file,
    usecols=usecols,
    parse_dates=["utc_timestamp"],
    chunksize=100_000
):
    chunk["year"] = chunk["utc_timestamp"].dt.year

    for (site, year), group in chunk.groupby(
        ["anonymised_data_centre_name", "year"]
    ):
        key = (site, year)

        if key not in unique_timestamps:
            unique_timestamps[key] = set()

        unique_timestamps[key].update(
            group["utc_timestamp"].dropna()
        )

# ============================================================
# 4. Final summary table
# ============================================================

final = rows.copy()

final["n_unique_timestamps"] = [
    len(unique_timestamps.get(
        (site, year), set()
    ))
    for site, year in zip(
        final["anonymised_data_centre_name"],
        final["year"]
    )
]

def expected_points(year):
    return (
        366 * 48
        if pd.Timestamp(year=year, month=1, day=1).is_leap_year
        else 365 * 48
    )

final["expected"] = final["year"].apply(expected_points)

final["complete"] = (
    final["n_unique_timestamps"]
    == final["expected"]
)

final.head()


In [ ]:
# ============================================================
# 5. Select the data centre and year
# ============================================================

selected_site = "Data Centre #13"
selected_year = 2025

# ============================================================
# 6. Extract the selected site + year
# ============================================================

output_file = (
    BASE_DIR /
    f"{selected_site.replace(' ', '_')}_{selected_year}.csv"
)

first_chunk = True
total_rows = 0

for chunk in pd.read_csv(
    csv_file,
    parse_dates=["utc_timestamp"],
    chunksize=100_000
):

    site_year = chunk[
        (chunk["anonymised_data_centre_name"] == selected_site) &
        (chunk["utc_timestamp"].dt.year == selected_year)
    ]

    if not site_year.empty:

        site_year = site_year.sort_values("utc_timestamp")

        site_year.to_csv(
            output_file,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False
        total_rows += len(site_year)

# ============================================================
# 7. Export result 1 datacenter/1 year with time steps = 30 mins
# ============================================================

print(f"\nExported: {output_file}")
print(f"Data Centre: {selected_site}")
print(f"Year: {selected_year}")
print(f"Rows: {total_rows}")


### Steps 4–8: Stage B — quality check, resample, scale to kW/kWh

In [ ]:
import numpy as np

# ============================================================
# 1. SETTINGS
# ============================================================

INPUT_FILE = output_file  # output of Stage A

# Reference capacity of the data centre.
# This is a modelling assumption because UKPN anonymises the real MIC.
P_REF_MW = 10.0

# Assumed power factor, used to convert kVA -> kW.
POWER_FACTOR = 0.95

YEAR = selected_year

EXPECTED_HALF_HOURLY = 365 * 48      # 17,520
EXPECTED_HOURLY = 365 * 24           # 8,760

OUTPUT_HOURLY = BASE_DIR / f"{selected_site.replace(' ', '_')}_{YEAR}_hourly.csv"
OUTPUT_QUALITY = BASE_DIR / f"{selected_site.replace(' ', '_')}_{YEAR}_quality_report.txt"

print("=" * 60)
print("DATA CENTRE LOAD PROFILE PROCESSING")
print("=" * 60)

print("\nLoading:")
print(INPUT_FILE)

df = pd.read_csv(INPUT_FILE, parse_dates=["utc_timestamp"])

print(f"Rows loaded: {len(df):,}")


In [ ]:
# ============================================================
# 2. BASIC CHECKS
# ============================================================

required_columns = ["utc_timestamp", "hh_utilisation_ratio"]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data = df[["utc_timestamp", "hh_utilisation_ratio"]].copy()
data = data.sort_values("utc_timestamp")

# ============================================================
# 3. DATA QUALITY CHECK
# ============================================================

data["year"] = data["utc_timestamp"].dt.year
years_found = sorted(data["year"].dropna().unique())
print(f"\nYears found: {years_found}")

if YEAR not in years_found:
    raise ValueError(f"No data found for year {YEAR}.")

data = data[data["year"] == YEAR].copy()
data = data.drop(columns=["year"])

# 3.1 Missing timestamps
expected_30min = pd.date_range(
    start=f"{YEAR}-01-01 00:00:00",
    end=f"{YEAR}-12-31 23:30:00",
    freq="30min"
)
actual_timestamps = data["utc_timestamp"]
missing_timestamps = expected_30min.difference(actual_timestamps)
extra_timestamps = actual_timestamps[~actual_timestamps.isin(expected_30min)].unique()

# 3.2 Duplicate timestamps
duplicate_mask = data["utc_timestamp"].duplicated(keep=False)
duplicate_rows = data[duplicate_mask].copy()
n_duplicate_rows = len(duplicate_rows)
n_duplicate_timestamps = duplicate_rows["utc_timestamp"].nunique() if n_duplicate_rows > 0 else 0

# 3.3 Row counts
n_rows = len(data)
n_unique_timestamps = data["utc_timestamp"].nunique()

print(f"\nRows:                 {n_rows:,}")
print(f"Unique timestamps:    {n_unique_timestamps:,}")
print(f"Expected timestamps:  {EXPECTED_HALF_HOURLY:,}")
print(f"\nMissing timestamps:   {len(missing_timestamps):,}")
print(f"Duplicate rows:       {n_duplicate_rows:,}")
print(f"Duplicate timestamps: {n_duplicate_timestamps:,}")
print(f"Extra timestamps:     {len(extra_timestamps):,}")

# 3.4 Utilisation values
util = data["hh_utilisation_ratio"]
n_missing_util = util.isna().sum()
n_negative_util = (util < 0).sum()
n_over_1_util = (util > 1).sum()

print(f"\nMissing utilisation:  {n_missing_util:,}")
print(f"Negative utilisation: {n_negative_util:,}")
print(f"Utilisation > 1:      {n_over_1_util:,}")
print("\nUtilisation statistics:")
print(util.describe())

# 3.5 30-minute frequency
timestamp_diff = data["utc_timestamp"].sort_values().diff()
unexpected_intervals = timestamp_diff[
    timestamp_diff.notna() & (timestamp_diff != pd.Timedelta(minutes=30))
]
print(f"\nUnexpected time intervals: {len(unexpected_intervals):,}")


In [ ]:
# ============================================================
# 4. STOP IF DATA QUALITY IS NOT SUITABLE
# ============================================================

if (
    len(missing_timestamps) > 0
    or n_duplicate_timestamps > 0
    or n_missing_util > 0
    or n_negative_util > 0
    or len(unexpected_intervals) > 0
):
    print("\nWARNING:")
    print("The half-hourly dataset is not perfectly continuous.")
    print("The script will NOT interpolate or fill missing data.")
    print("\nYou should inspect the quality issues before using this profile for BHESS.")

    with open(OUTPUT_QUALITY, "w", encoding="utf-8") as f:
        f.write("DATA CENTRE QUALITY REPORT\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Year: {YEAR}\n")
        f.write(f"Rows: {n_rows}\n")
        f.write(f"Unique timestamps: {n_unique_timestamps}\n")
        f.write(f"Expected timestamps: {EXPECTED_HALF_HOURLY}\n")
        f.write(f"\nMissing timestamps: {len(missing_timestamps)}\n")
        f.write(f"Duplicate timestamps: {n_duplicate_timestamps}\n")
        f.write(f"Missing utilisation: {n_missing_util}\n")
        f.write(f"Negative utilisation: {n_negative_util}\n")
        f.write(f"Utilisation > 1: {n_over_1_util}\n")
        f.write(f"Unexpected intervals: {len(unexpected_intervals)}\n")

        if len(missing_timestamps) > 0:
            f.write("\nMISSING TIMESTAMPS\n")
            f.write("-" * 60 + "\n")
            for ts in missing_timestamps:
                f.write(str(ts) + "\n")

    print(f"\nQuality report saved to:\n{OUTPUT_QUALITY}")
    raise SystemExit("\nProcessing stopped because the input data requires inspection.")


Each hour contains two half-hour observations. The hourly utilisation is the arithmetic mean of the two values.

\[
u_h = \frac{u_{h,1}+u_{h,2}}{2}
\]

This produces 8,760 hourly observations for a complete 2025 dataset.

In [ ]:
# ============================================================
# 5. 30 MINUTES -> 1 HOUR
# ============================================================

data = data.set_index("utc_timestamp")

hourly = (
    data["hh_utilisation_ratio"]
    .resample("1h")
    .mean()
    .to_frame(name="hourly_utilisation_ratio")
)

print(f"\nHourly points generated: {len(hourly):,}")

if len(hourly) != EXPECTED_HOURLY:
    raise ValueError(f"Expected {EXPECTED_HOURLY} hourly points, but obtained {len(hourly)}.")



The UKPN utilisation ratio is converted into apparent power using the assumed reference capacity.

For this example:

\[
P_{ref}=10\,MW=10,000\,kW
\]

Then:

\[
S_t[kVA]=u_t\times10,000
\]

and using the assumed power factor:

\[
P_t[kW]=S_t\times0.95
\]

**Important:** this is a reconstructed/scaled load profile, not the measured absolute kW profile of the anonymised data centre, because the actual MIC is not available.

In [ ]:
# ============================================================
# 6. CONVERT TO POWER / ENERGY
# ============================================================

P_REF_KW = P_REF_MW * 1000
print(f"\nReference capacity: {P_REF_MW:.2f} MW ({P_REF_KW:,.0f} kW)")
print(f"Power factor:       {POWER_FACTOR:.3f}")

# S(t) = utilisation(t) x Maximum Import Capacity (assumed = P_REF)
hourly["apparent_power_kVA"] = hourly["hourly_utilisation_ratio"] * P_REF_KW

# P = S x power factor
hourly["load_kW"] = hourly["apparent_power_kVA"] * POWER_FACTOR

# Time step = 1h => E[kWh] = P[kW] x 1[h]
hourly["load_kWh"] = hourly["load_kW"] * 1.0

hourly["hour"] = np.arange(1, len(hourly) + 1)

hourly = hourly[
    ["hour", "hourly_utilisation_ratio", "apparent_power_kVA", "load_kW", "load_kWh"]
]

hourly.head()

In [ ]:
# ============================================================
# 7. FINAL CHECK (8760 HOURS)
# ============================================================

print(f"\nNumber of hours: {len(hourly):,}")
print(f"Expected hours:  {EXPECTED_HOURLY:,}")
print(f"Minimum load:    {hourly['load_kW'].min():,.2f} kW")
print(f"Maximum load:    {hourly['load_kW'].max():,.2f} kW")
print(f"Mean load:       {hourly['load_kW'].mean():,.2f} kW")
print(f"Annual energy:   {hourly['load_kWh'].sum():,.2f} kWh")
print(f"Annual energy:   {hourly['load_kWh'].sum() / 1000:,.2f} MWh")
print(f"Load factor:     {hourly['load_kW'].mean() / hourly['load_kW'].max():.4f}")

# ============================================================
# 8. EXPORT
# ============================================================

hourly.to_csv(OUTPUT_HOURLY, index=False)

print(f"\nHourly load profile saved to:\n{OUTPUT_HOURLY}")


## 4. Example

The resulting CSV contains one row per hour and can be used as the demand input for an energy-system model.
Example run with **Data Centre #13, year 2025**, `P_REF_MW = 10.0`, `POWER_FACTOR = 0.95`:

```
Years found: [2025]

Rows:                 17,520
Unique timestamps:    17,520
Expected timestamps:  17,520

Missing timestamps:   0
Duplicate rows:       0
Extra timestamps:     0

Hourly points generated: 8,760

Reference capacity: 10.00 MW (10,000 kW)
Power factor:       0.950

Number of hours: 8,760
Minimum load:    0.00 kW
Maximum load:    9,196.84 kW
Mean load:       3,711.52 kW
Annual energy:   32,512,915.20 kWh
Annual energy:   32,512.92 MWh
Load factor:     0.4035

Hourly load profile saved to:
Data_Centre_#13_2025_hourly.csv
```

Output file (`Data_Centre_#13_2025_hourly.csv`):

| hour | hourly_utilisation_ratio | apparent_power_kVA | load_kW | load_kWh |
|---|---|---|---|---|
| 1 | 0.2548 | 2,548.0 | 2,420.6 | 2,420.6 |
| 2 | 0.2460 | 2,460.3 | 2,337.3 | 2,337.3 |
| ... | ... | ... | ... | ... |
| 8760 | 0.3012 | 3,012.4 | 2,861.8 | 2,861.8 |

This file can be used directly as an input load profile for LP/MILP sizing models (e.g. PV–battery–hydrogen), with `P_REF_MW` adjusted to match the scale of the data centre you want to study.

## Data source & attribution

The shape of the load profile is derived from UK Power Networks' open dataset, *"Data Centre Demand Profiles"* (check the current licence terms on the source page before redistributing). This repository does not redistribute the raw UKPN file, only the *processing code*.

## Important note

Absolute power/energy values (`load_kW`, `load_kWh`) depend on the user-chosen `P_REF_MW` and `POWER_FACTOR` — these are **modelling assumptions**, not measured values (UKPN anonymises each site's real capacity). Only the **shape** of the load over time (relative utilisation across the year) is genuinely derived from real metered data.